# Chapter 1 — TD-PINNs on **2-D** Burgers (just for the hahas)

Quick check that the framework handles a 2-D problem. Scalar 2-D viscous Burgers on $[0,1]^2$:

$$u_t + u\,(u_x + u_y) = \nu\,(u_{xx} + u_{yy}).$$

**Exact solution (ground truth).** A travelling diagonal front

$$u(x,y,t) = \frac{1}{1 + \exp\!\big(\frac{x+y-c-t}{2\nu}\big)}$$

solves it exactly (verify by substitution: $u_t=u(1-u)/2\nu$, $u_x=u_y=-u(1-u)/2\nu$, etc.). The front
sits on $x+y = c+t$ and moves along the diagonal.

**Why no Dirichlet BC.** The front *moves*, so the true boundary values change in time, but the
framework's TD-PINNs applies a single fixed `u_bc` for all steps — imposing the $t{=}0$ values would be
wrong later. The clean fix that *works here* is to drop the spatial BC (`bc_weight = 0`) and let the
**initial condition + PDE residual + Crank–Nicolson step-continuity** determine the evolution. (Imposing
the stale $t{=}0$ BC roughly triples the final $L_\infty$ — confirmed before writing this notebook.)

In [ ]:
import sys, os, time

ROOT = '/home/df.gomezb/Tesis'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pinns_framework import (
    PINArchitecture, PINSpatialDimension, PINLossFunction, PINTrainer,
    compute_metrics, collocation_uniform,
)

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    'font.family'     : 'serif',
    'font.serif'      : ['DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size'       : 12,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 11,
})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIGDIR = 'figs'; os.makedirs(FIGDIR, exist_ok=True)
print('torch:', torch.__version__, '| device:', DEVICE)

## 1. Problem setup — exact travelling front

In [ ]:
NU = 0.1
L  = 1.0
C  = 0.5            # initial front position: x + y = C

def exact_np(t, x, y):
    return 1.0 / (1.0 + np.exp((x + y - C - t) / (2 * NU)))

gx = np.linspace(0, L, 120)
XX, YY = np.meshgrid(gx, gx, indexing='ij')
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, t in zip(axes, [0.0, 0.25, 0.5]):
    im = ax.imshow(exact_np(t, XX, YY).T, origin='lower', extent=[0, L, 0, L],
                   vmin=0, vmax=1, cmap='viridis')
    ax.set_title(f't = {t:.2f}'); ax.set_xlabel('x'); ax.set_ylabel('y')
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Exact 2-D Burgers front  u(x,y,t)', fontsize=13)
plt.tight_layout(); plt.show()

## 2. Configuration

In [ ]:
PARAMS_ARCH = {'layers': (2, 64, 64, 64, 1), 'activation': 'tanh'}   # input (x, y)

DT     = 0.1
T_END  = 0.5
NT     = round(T_END / DT)
times  = [k * DT for k in range(NT + 1)]

EPOCHS     = 8000
TOLERANCE  = 1e-4
LR, MAX_LR, MIN_LR = 5e-3, 5e-3, 1e-6
IC_WEIGHT  = 10.0
BC_WEIGHT  = 0.0      # no spatial Dirichlet BC (front-driven, time-varying)
PDE_WEIGHT = 1.0
N_COLLOC   = 3000     # interior collocation points (2-D)
N_EVAL     = 150      # per-axis -> N_EVAL^2 evaluation grid

print(f'{NT} steps, dt={DT}, T={T_END}  |  front sweeps x+y from {C} to {C+T_END}')

## 3. IC, 2-D Crank–Nicolson residual, exact evaluator

$S(w) = w\,(w_x + w_y) - \nu\,(w_{xx} + w_{yy})$; the residual is $\frac{u-u^{t-1}}{\Delta t} + \tfrac12[S(u)+S(u^{t-1})]$.

In [ ]:
x_dim = PINSpatialDimension('x', domain=(0.0, L), n_colloc=N_COLLOC)
y_dim = PINSpatialDimension('y', domain=(0.0, L), n_colloc=N_COLLOC)

# IC sampled from the exact solution on a grid
g_ic = np.linspace(0, L, 51)
Xi, Yi = np.meshgrid(g_ic, g_ic, indexing='ij')
pts_ic = np.stack([Xi.ravel(), Yi.ravel()], axis=1)
x_ic = torch.tensor(pts_ic, dtype=torch.float32)
u_ic = torch.tensor(exact_np(0, pts_ic[:, 0], pts_ic[:, 1]), dtype=torch.float32).unsqueeze(1)

# BC placeholders (unused since BC_WEIGHT = 0, but fit() requires the arguments)
x_bc = torch.tensor([[0.0, 0.0], [L, L]], dtype=torch.float32)
u_bc = torch.zeros(2, 1, dtype=torch.float32)

def burgers2d_residual(u, u_prev, x, dt, nu):
    def S(w):
        g    = torch.autograd.grad(w.sum(), x, create_graph=True)[0]   # (N, 2)
        w_x  = g[:, 0:1]; w_y = g[:, 1:2]
        w_xx = torch.autograd.grad(w_x.sum(), x, create_graph=True)[0][:, 0:1]
        w_yy = torch.autograd.grad(w_y.sum(), x, create_graph=True)[0][:, 1:2]
        return w * (w_x + w_y) - nu * (w_xx + w_yy)
    return (u - u_prev.detach()) / dt + 0.5 * (S(u) + S(u_prev).detach())

loss_fn = PINLossFunction([x_dim, y_dim], burgers2d_residual)

def exact_fn(t, xy):
    p = xy.detach().cpu().numpy()
    u = exact_np(t, p[:, 0], p[:, 1])
    return torch.tensor(u, dtype=torch.float32, device=xy.device).unsqueeze(1)

## 4. Train TD-PINNs (Crank–Nicolson, 2-D)

In [ ]:
torch.manual_seed(0); np.random.seed(0)
net = PINArchitecture('MLP', PARAMS_ARCH)
trainer = PINTrainer(
    net=net, scheme='TD-PINNS', loss_fn=loss_fn, spatial_dims=[x_dim, y_dim],
    epochs=EPOCHS, dt=DT, lr=LR, max_lr=MAX_LR, min_lr=MIN_LR, tolerance=TOLERANCE,
    ic_weight=IC_WEIGHT, bc_weight=BC_WEIGHT, pde_weight=PDE_WEIGHT,
    collocation_fn=collocation_uniform, device=DEVICE, verbose=True, log_every=2000,
)
t0 = time.perf_counter()
nets = trainer.fit(x_ic=x_ic, u_ic=u_ic, x_bc=x_bc, u_bc=u_bc, T=T_END, nu=NU)
wall = time.perf_counter() - t0
print(f'\ndone | per-step epochs = {[h["epochs"] for h in trainer.step_history]} | wall {wall:.1f}s')

## 5. Errors vs exact — $L_1$, $L_2$, $L_\infty$ per time level

Errors are integrated on a dense $N_{eval}^2$ grid by `compute_metrics`; $L_1$ (mean-abs) is added.

In [ ]:
df = compute_metrics(nets, times, [x_dim, y_dim], n_eval=N_EVAL,
                     exact_fn=exact_fn, device=DEVICE)

# add L1 per level (compute_metrics returns L2 / Linf)
from pinns_framework import make_eval_grid
xe = make_eval_grid([x_dim, y_dim], N_EVAL).to(DEVICE)
l1 = []
for t, net_t in zip(times, nets):
    net_t.eval()
    with torch.no_grad():
        up = net_t(xe)
        if up.dim() == 1:
            up = up.unsqueeze(1)
    l1.append(float((up - exact_fn(t, xe)).abs().mean()))
df['L1'] = l1

pd.set_option('display.float_format', '{:.4e}'.format)
print(df[['L1', 'L2', 'Linf']].to_string())
print(f'\nmean  L1={df["L1"].mean():.3e}  L2={df["L2"].mean():.3e}  Linf={df["Linf"].mean():.3e}')

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for col, mk, lab in [('L1', 'o', r'$L_1$'), ('L2', 's', r'$L_2$'), ('Linf', '^', r'$L_\infty$')]:
    ax.semilogy(times, df[col].values, '-', marker=mk, ms=6, lw=1.5, label=lab)
ax.set_xlabel('t'); ax.set_ylabel('error vs exact')
ax.set_title('2-D Burgers (TD-PINNs / CN) — error per time level')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'burgers2d_errors.pdf'), bbox_inches='tight')
print('saved ->', os.path.join(FIGDIR, 'burgers2d_errors.pdf'))
plt.show()

## 6. 2-D field comparison — exact vs predicted vs error

In [ ]:
ng = 120
gv = np.linspace(0, L, ng)
GX, GY = np.meshgrid(gv, gv, indexing='ij')
grid = torch.tensor(np.stack([GX.ravel(), GY.ravel()], axis=1), dtype=torch.float32).to(DEVICE)

def pred_field(k):
    net_k = nets[k].eval()
    with torch.no_grad():
        return net_k(grid).cpu().numpy().reshape(ng, ng)

show_idx = [NT // 2, NT]          # mid and final time
fig, axes = plt.subplots(len(show_idx), 3, figsize=(13, 4.2 * len(show_idx)))
for row, k in enumerate(show_idx):
    t = times[k]
    u_ex = exact_np(t, GX, GY)
    u_pr = pred_field(k)
    err  = np.abs(u_pr - u_ex)
    for col, (field, title, kw) in enumerate([
        (u_ex, f'exact  (t={t:.2f})',      dict(vmin=0, vmax=1, cmap='viridis')),
        (u_pr, f'TD-PINNs  (t={t:.2f})',   dict(vmin=0, vmax=1, cmap='viridis')),
        (err,  f'|error|  (t={t:.2f})',    dict(cmap='magma')),
    ]):
        ax = axes[row, col]
        im = ax.imshow(field.T, origin='lower', extent=[0, L, 0, L], **kw)
        ax.set_title(title); ax.set_xlabel('x'); ax.set_ylabel('y')
        fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('2-D Burgers: exact vs TD-PINNs prediction vs pointwise error', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIGDIR, 'burgers2d_fields.pdf'), bbox_inches='tight')
print('saved ->', os.path.join(FIGDIR, 'burgers2d_fields.pdf'))
plt.show()

## 7. Notes

- The framework handles 2-D out of the box: two `PINSpatialDimension`s, `collocation_uniform` samples
  in 2-D, and `compute_metrics` integrates on the $N_{eval}^2$ grid.
- Time integrator is Crank–Nicolson in the residual (the spatial operator now includes both $x$ and $y$
  derivatives); `u_prev` keeps its graph so CN can differentiate the frozen step.
- **No spatial BC** (`bc_weight = 0`): the moving front makes a fixed Dirichlet BC inconsistent, so the
  solution is anchored by the IC + PDE + step-continuity. The error grows mildly in time as the front
  approaches the (unconstrained) boundary — visible in §5/§6.
- Figures: `figs/burgers2d_errors.pdf`, `figs/burgers2d_fields.pdf`.
- To sharpen the front, lower `NU`; you may then need more `N_COLLOC` / `EPOCHS`.